# PO Delay Root Cause Analyzer: Clasificación por Etapa (reglas de negocio)

**Objetivo de Fase 2:** clasificar cada PO retrasado por la etapa del lifecycle responsable del retraso, con reglas reproducibles y umbrales externalizados en `rules_config.json`.

**Input:** el DataFrame limpio que entrega `clean_po_data()` de `pipeline_core` (Fase 1, congelada).

**Lógica:** toda en funciones reusables de `classifier_core.py` (clasificación + severidad + persistencia) y `metrics_core.py` (validación). **Este notebook solo PRESENTA**: llama a esas funciones, no reimplementa.

La metodología completa (decisiones del mentor, umbrales y su porqué) vive en el [README de la fase](README.md).

---

### 1. Setup & Imports

In [ ]:
# El clasificador (Fase 2) y el pipeline de limpieza (Fase 1, su clean_po_data
# entra como INPUT). Ambas carpetas empiezan con dígito, así que no son paquetes
# importables por su nombre: las añadimos a sys.path resolviendo desde el cwd del
# notebook (su carpeta 02_clasif_reglas_negocio) hacia la raíz del repo.
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "02_clasif_reglas_negocio":
    REPO_ROOT = REPO_ROOT.parent
for _sub in ("01_data_pipeline_and_eda", "02_clasif_reglas_negocio"):
    _p = str(REPO_ROOT / _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pipeline_core import clean_po_data
from classifier_core import classify_po_stages, load_rules_config
import metrics_core as mc

# Umbrales externalizados (confirmados por el mentor: carrier 8h / yard 4h / dock 6h).
rules = load_rules_config()

# CSV local (gitignored): respeta PO_CSV_PATH; si no, el default convencional.
_env = os.environ.get("PO_CSV_PATH")
CSV = Path(_env) if _env else REPO_ROOT / "data" / "raw" / "po_root_cause_synthetic.csv"

df_clean = clean_po_data(pd.read_csv(CSV, low_memory=False))
df = classify_po_stages(df_clean, rules)
print(f"Clasificados {len(df)} POs · {(df['delay_days_calc'] > 0).sum()} tardíos")
rules["thresholds"]

### 2. Reparto de la clasificación (sobre tardíos)

`stage_primary` por etapa responsable, `severity` determinística y la subclase `dc_substage`.

In [ ]:
tardios = df[df["delay_days_calc"] > 0]

print("stage_primary (tardíos):")
print((tardios["stage_primary"].value_counts(normalize=True) * 100).round(1).to_string())

print("\nseverity (tardíos):")
print(tardios["severity"].value_counts().to_string())

print("\ndc_substage (solo etapa DC):")
print(df.loc[df["stage_primary"] == "DC", "dc_substage"].value_counts().to_string())

### 3. Validación: Stage accuracy (#46) y Reason agreement (#47)

`metrics_core` valida la clasificación contra dos referencias independientes. **Stage accuracy** compara `stage_primary` (exceso sobre umbral) con el *gap dominante* (duración bruta del tramo más largo) — métricas distintas a propósito. **Reason agreement** la compara con la anotación humana (`REASON_DSC`).

In [ ]:
acc = mc.stage_accuracy(df)
print(f"#46 Stage accuracy: {acc['accuracy']:.1%}  (umbral >80%: {'PASA' if acc['passes'] else 'NO'})")
print(f"   evaluables={acc['n_evaluables']}  confiables={acc['n_confiables']}  tardíos={acc['n_tardios']}")
print("\n   stage_primary (filas) vs gap_dominante (cols):")
print(acc["matriz"].to_string())

ra = mc.reason_agreement(df)
print(f"\n#47 Reason agreement: {ra['agreement']:.1%}  "
      f"(clasificables={ra['n_clasificable']}, mismatches={ra['n_mismatches']}, "
      f"REASON_DSC nulos en tardíos={ra['n_reason_null']})")

### 4. Sensibilidad del umbral de carrier (4 / 6 / 8 / 12 h)

El mentor pidió mostrar cómo varía el % de carrier con el umbral. Se reclasifica inyectando cada umbral en `rules` (sin tocar el módulo). Nota: el umbral mueve mucho la *flag bruta* `flag_carrier_calc`, pero poco `stage_primary`, porque con vendor por STA push la señal de vendor domina el argmax.

In [ ]:
import copy

filas = []
for thr in (4.0, 6.0, 8.0, 12.0):
    r = copy.deepcopy(rules)
    r["thresholds"]["carrier_lag_hrs"]["value"] = thr
    d = classify_po_stages(df_clean, r)
    t = d[d["delay_days_calc"] > 0]
    flag = d["flag_carrier_calc"].where(d["_carrier_medible"], False)
    filas.append({
        "umbral_h": thr,
        "flag_carrier_%": round(100 * flag.mean(), 1),
        "Vendor_%":  round(100 * (t["stage_primary"] == "Vendor").mean(), 1),
        "Carrier_%": round(100 * (t["stage_primary"] == "Carrier").mean(), 1),
        "DC_%":      round(100 * (t["stage_primary"] == "DC").mean(), 1),
        "Indet_%":   round(100 * (t["stage_primary"] == "Indeterminado").mean(), 1),
    })

pd.DataFrame(filas).set_index("umbral_h")

### 5. Mismatches (#47): la tesis del proyecto

Casos donde el cómputo temporal contradice la anotación humana (`REASON_DSC`). Ordenados por fuerza de señal: la evidencia de que los timestamps son más precisos que el reason code. Están disponibles como posible few-shot para Fase 3. **Estado actual: el prompt de F3 es zero-shot — todavía no los consume; cablearlos es decisión de diseño de prompt, pendiente en F3.**

In [ ]:
mismatches = mc.select_mismatches(df, n=8)
mismatches[["PO_NBR", "stage_primary", "reason_group_manual", "REASON_DSC",
            "senal_computo_hrs", "delay_days_calc"]]

### 6. Persistencia (#49)

`save_classified_output` escribe el veredicto a `data/processed/` (gitignored; se regenera ejecutando). Reusable por Fase 3/4.

In [ ]:
from classifier_core import save_classified_output

out_path = save_classified_output(df)
print(f"Escrito: {out_path}")